# 02 — Do vetor ao índice de busca

A etapa 01 gravou textos, metadados e embeddings normalizados. Agora esses vetores serão inseridos em uma coleção persistente do Chroma. Em seguida, será comparada a busca aproximada HNSW com uma busca exata e será estudada a seleção MMR para reduzir resultados repetidos. A saída é o banco que a etapa 03 consultará para montar o RAG.

```mermaid
flowchart LR
 A[dados_processados: chunks + vetores] --> B[Chroma persistente]
 B --> C[consulta: embedding da pergunta]
 C --> D[HNSW: candidatos aproximados]
 D --> E[ranking por cosseno]
 E --> F[MMR: diversidade opcional]
 F --> G[trechos para o RAG]
```

In [1]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: 02 — Banco vetorial: Chroma, HNSW, cosseno e MMR
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from pathlib import Path
import json

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

Projeto: d:\Projetos_v2\Agente_IA_Transformers_Credito


## 1. Carregar os vetores e preparar a consulta

A célula anterior localizou a pasta do projeto. Agora carregam-se manifesto, textos e embeddings produzidos na etapa 01 e o mesmo encoder para representar uma pergunta de consulta. A checagem de dimensões e alinhamento confirma que cada vetor corresponde a um chunk.

Para vetores normalizados, $q^T x$ é a similaridade cosseno, e a distância cosseno é $1-q^T x$. A consulta também deve ser normalizada. Similaridade indica proximidade geométrica no espaço do encoder; não é probabilidade de verdade.
\n
\n
---\n
\n
### Intuição técnica e exemplo\n
\n
### Similaridade de cosseno: equação e conta

A similaridade de cosseno compara a direção de dois vetores:

$$
\operatorname{sim}_{cos}(q,x)=\frac{q^T x}{\lVert q\rVert_2\lVert x\rVert_2},
\qquad
d_{cos}(q,x)=1-\operatorname{sim}_{cos}(q,x).
$$

Exemplo didático: q = [1, 0], a = [0,8; 0,6] e b = [0, 1]. Os três vetores têm norma 1. Assim, sim(q,a) = 0,8 e sim(q,b) = 0. As distâncias são 0,2 e 1; portanto, a fica mais perto da consulta. O valor 0,8 é proximidade geométrica, não 80% de chance de a resposta estar correta.

In [2]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: 02 — Banco vetorial: Chroma, HNSW, cosseno e MMR
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
import hashlib
import json
import numpy as np
from sentence_transformers import SentenceTransformer

manifesto = json.loads((DIRETORIO_PROCESSADOS / "manifesto_corpus.json").read_text(encoding="utf-8"))
chunks = [json.loads(linha) for linha in (DIRETORIO_PROCESSADOS / "chunks.jsonl").read_text(encoding="utf-8").splitlines() if linha]
vetores = np.load(DIRETORIO_PROCESSADOS / "embeddings.npy")
NOME_ENCODER = manifesto["encoder"]
assert len(chunks) == len(vetores) == manifesto["quantidade_chunks"]
assert np.allclose(np.linalg.norm(vetores[:20], axis=1), 1.0, atol=1e-4)
encoder = SentenceTransformer(NOME_ENCODER, device="cpu")
print("Chunks e embeddings carregados:", vetores.shape)

c:\Users\leova\anaconda3\envs\engenharia_ia_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9467.42it/s]


Chunks e embeddings carregados: (990, 384)


## 2. Criar ou atualizar a coleção persistente

A coleção Chroma guarda texto, metadados e vetor sob IDs estáveis. O índice HNSW organiza conexões entre vetores para encontrar candidatos sem comparar a consulta com todos os registros. `space="cosine"` fixa a métrica. HNSW é aproximado: pode economizar trabalho de busca e ainda deixar passar algum vizinho exato.

IDs estáveis tornam a reexecução idempotente: o mesmo chunk atualiza o mesmo registro, em vez de criar cópias. Se o conteúdo de uma fonte mudou, os IDs antigos daquela fonte precisam ser removidos. A coleção persistida deve ser reaberta para confirmar que os dados sobreviveram ao processo.\n
\n
---\n
\n
### Intuição técnica e exemplo\n
\n
### Intuição do HNSW

HNSW organiza os vetores em um grafo de vizinhos com níveis: níveis superiores têm poucas conexões e ajudam a saltar para regiões promissoras; níveis inferiores refinam a busca local. Em vez de comparar a pergunta com todos os 990 vetores, o algoritmo navega por uma parte do grafo. Por ser aproximado, pode deixar de visitar o vizinho exato.

Dois controles têm papéis diferentes: k (ou n_results) define quantos resultados serão devolvidos; ef_search controla quantos candidatos o índice explora para encontrá-los. Aumentar ef_search tende a elevar o esforço e pode melhorar a recuperação de vizinhos, mas não muda k. A configuração deve ser avaliada em consultas representativas.

### Exemplo de navegação (apenas conceitual)

Imagine pontos A, B, C e D espalhados num plano. A busca começa em um ponto de uma camada esparsa; se B estiver mais perto da pergunta do que A, a navegação segue para B. Ao descer para uma camada mais densa, examina vizinhos próximos de B e refina os candidatos. A busca exata, em contraste, calcula a distância até A, B, C e D. Esse exemplo mostra a lógica do grafo, não reproduz o algoritmo completo nem promete que o vizinho mais próximo será encontrado.

In [3]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Parte III — Banco vetorial real
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
import chromadb
cliente = chromadb.PersistentClient(path=str(DIRETORIO_PROJETO / "banco_chroma"))
colecao = cliente.get_or_create_collection(
    "transformers_credito_v1",
    configuration={"hnsw": {"space": "cosine", "max_neighbors": 16, "ef_construction": 100, "ef_search": 100}},
    metadata={"encoder": NOME_ENCODER},
)
assert colecao.metadata["encoder"] == NOME_ENCODER
ids_atuais = [c["id"] for c in chunks]
for inicio in range(0, len(chunks), 100):
    lote = chunks[inicio:inicio+100]
    colecao.upsert(
        ids=[c["id"] for c in lote], documents=[c["texto"] for c in lote],
        embeddings=vetores[inicio:inicio+100].tolist(),
        metadatas=[{k:v for k,v in c.items() if k not in ["id", "texto"]} for c in lote],
    )
obsoletos = sorted(set(colecao.get()["ids"]) - set(ids_atuais))
if obsoletos:
    colecao.delete(ids=obsoletos)
assert colecao.count() == len(chunks)
print("Registros persistidos:", colecao.count())
print("Configuração:", colecao.configuration)

Registros persistidos: 990
Configuração: {'hnsw': {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction object at 0x000002420EC8BA90>}


## 3. Comparar busca exata e HNSW; depois observar MMR

A busca exata calcula a similaridade da pergunta com todos os vetores e serve como referência do ranking para essa métrica. HNSW retorna uma aproximação. O (Recall_{ANN}@k) mede a fração dos k vizinhos exatos que também apareceu no resultado aproximado; não mede se os trechos são bons para responder à pergunta.

MMR atua depois de recuperar candidatos. Em cada passo, escolhe um trecho que seja relevante para a pergunta e, ao mesmo tempo, diferente dos trechos já escolhidos. O parâmetro de equilíbrio controla esse compromisso. A comparação deve mostrar os textos lado a lado: maior diversidade pode reduzir repetição, mas também pode retirar uma passagem complementar importante.\n
\n
---\n
\n
### Intuição técnica e exemplo\n
\n
### Busca exata e Recall ANN@k

A busca exata calcula a distância da consulta a todos os vetores elegíveis e ordena o resultado. HNSW é comparado com esse ranking:

$$
\operatorname{Recall}_{ANN}@k=
\frac{|IDs_{HNSW}@k\cap IDs_{exatos}@k|}{k}.
$$

Exemplo: os três vizinhos exatos são {A, B, C}; HNSW retorna {A, C, D}. A interseção contém dois IDs, então Recall ANN@3 = 2/3. Isso mede o quanto a aproximação reproduz a busca exata; não mede se A, B ou C são evidências úteis para responder.

In [4]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
import sys
import inspect
if str(DIRETORIO_PROJETO) not in sys.path:
    sys.path.insert(0, str(DIRETORIO_PROJETO))
from rag import selecionar_mmr, recuperar, montar_mensagens
print(inspect.getsource(selecionar_mmr))

pergunta_rag = "Como a atenção pode representar sequências e quais cuidados existem na previsão de inadimplência de consumidores?"
vetor_consulta = encoder.encode(pergunta_rag, normalize_embeddings=True)
similaridades = vetores @ vetor_consulta
indices_exatos = np.argsort(-similaridades)[:4]
resultado_ann = colecao.query(query_embeddings=[vetor_consulta.tolist()], n_results=4)
ids_exatos = {chunks[i]["id"] for i in indices_exatos}
recall_ann = len(ids_exatos.intersection(resultado_ann["ids"][0])) / 4
print("Recall ANN@4 desta consulta:", recall_ann)
trechos = recuperar(colecao, encoder, pergunta_rag, usar_mmr=True, k=4, fetch_k=12)
for trecho in trechos:
    print(trecho["source_id"], "página", trecho["page"], trecho["texto"][:250])

def selecionar_mmr(vetores, consulta, k=4, peso=0.7):
    """Seleciona candidatos normalizados por relevância menos redundância."""
    vetores = np.asarray(vetores, dtype=np.float32)
    consulta = np.asarray(consulta, dtype=np.float32)
    if len(vetores) == 0:
        return []
    relevancia = vetores @ consulta
    escolhidos = [int(np.argmax(relevancia))]
    while len(escolhidos) < min(k, len(vetores)):
        redundancia = (vetores @ vetores[escolhidos].T).max(axis=1)
        scores = peso * relevancia - (1 - peso) * redundancia
        scores[escolhidos] = -np.inf
        escolhidos.append(int(np.argmax(scores)))
    return escolhidos

Recall ANN@4 desta consulta: 1.0
attention_is_all_you_need página 2 ]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2.
consumer_default página 19 

## 4. Checkpoint: o que a etapa 03 receberá

Ao fim desta etapa, a coleção persistida deve devolver IDs, distâncias, textos e metadados rastreáveis. A etapa 03 usará esses resultados como contexto do Qwen. Se uma resposta estiver errada, será possível verificar separadamente se o trecho correto não foi recuperado ou se a geração não o utilizou.

Uma consulta de demonstração e recall ANN perfeito em poucos vetores não formam um benchmark geral. Para escolher parâmetros, serão necessárias várias perguntas anotadas e medições de latência/memória.\n
\n
---\n
\n
### Intuição técnica e exemplo\n
\n
### Do índice para a geração

A saída do retriever precisa trazer mais que números: cada ID deve apontar para texto, título e página. A etapa 03 usará esses campos para compor o contexto do Qwen e mostrar referências. Dessa forma, uma resposta pode ser auditada voltando da afirmação ao trecho e do trecho à página do PDF.